# GSK_HepG2 ablation: metric vs training-data fraction

Pulls the single run from the `ablation_gsk_hepg2` wandb project, recovers the per-(fraction, model) rows from its history, and plots one line per model.

Each `wandb.log(row)` call in the ablation became a separate history step, so we read the full history with `scan_history()` and pivot on the `model` column ourselves.

In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = "ablations"
RUN_PREFIX = "gsk_hepg2_data_fraction"
RUN_NAME = None  # set to an exact run name to pin a specific run; otherwise use the latest
METRIC = "test_average_precision"  # swap for test_mcc, test_f1, test_roc_auc, ...

api = wandb.Api()
runs = api.runs(f"{api.default_entity}/{PROJECT}")
if RUN_NAME is not None:
    run = next(r for r in runs if r.name == RUN_NAME)
else:
    # Runs are named '<prefix>_<timestamp>'; pick the most recent matching run.
    matches = [r for r in runs if r.name.startswith(RUN_PREFIX)]
    run = max(matches, key=lambda r: r.created_at)
run.name, run.id

In [ ]:
# Full, unsampled history -> one row per (fraction, model).
df = pd.DataFrame(run.scan_history())
df = df[["fraction", "model", "n_train", METRIC]].sort_values(["model", "fraction"])
df

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for model, g in df.groupby("model"):
    g = g.sort_values("fraction")
    ax.plot(g["fraction"], g[METRIC], marker="o", label=model)

ax.set_xlabel("Training data fraction")
ax.set_ylabel(METRIC)
ax.set_title("GSK_HepG2 ablation")
ax.legend()
fig.tight_layout()
fig.savefig("gsk_hepg2_ablation.png", dpi=150)
plt.show()